## 离线读者模式：构造的金融数据描述分析

当前代码只用 Seed 101 构造的 180 个月月度面板演示方法，不运行 French 观测分支。原历史报告、图与练习答案是独立的阅读材料。French 原始、提取和处理数据未随包分发，所需权利与允许用途尚未明确。

按[读者导读](../../../../../projects/project01/READER_GUIDE.md)从包根运行。

# 项目1：金融数据分析

先确认数据对象、月度区间和小数单位，再计算当前构造面板的财富、风险与固定分段。列名NoDur、HiTec、Enrgy是方法示例标签，不是French行业观测。2010–2019是当前构造主表的120月；三个60月切片解释窗口敏感性，不能称为市场样本外检验。

[历史报告](../../../../../projects/project01/research_report/report.md)使用2026-10-05 French快照；[六题](../../../../../projects/project01/exercises.md)/[完整答案](../../../../../projects/project01/solutions.md)中只有第2、5题读取该历史报告，其余按独立给定输入手算或解释图轴。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project01_financial_data_analysis_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
PROJECT = ROOT / "projects/project01"
sys.path.insert(0, str(PROJECT))
from project01_analysis import load_config, make_project_figure, summarize_all_windows
from quant_course.performance import drawdown_path, wealth_index


## 1. 当前构造输入与历史快照的区别

下一代码单元读取共同配置，Seed101生成180个月的三列构造简单收益，RF固定0.001，并写constructed_monthly_returns.csv到刚打印的RUN_DIR。没有调用load_project_data读取French。

原历史工作路线才比对数据清单、核验ZIP/提取/处理CSV哈希并连接French与RF；这些说明保留在[字典](../../../../../projects/project01/data_dictionary.md)，哈希不证明历史可用时刻，也不授予当前执行许可。

In [ ]:
config = load_config(PROJECT / "configs/analysis.json")
# CONSTRUCTED: synthetic monthly returns; labels are method examples, not French observations.
rng = np.random.default_rng(101)
index = pd.period_range("2010-01", "2024-12", freq="M", name="month")
data = pd.DataFrame(rng.normal(0.004, 0.035, (len(index), 3)), index=index,
                    columns=[x["column"] for x in config["industries"]])
data["RF"] = 0.001
print("CONSTRUCTED panel:", len(data), "continuous months; French branch SKIPPED: rights unknown")
assert not data.isna().any().any()
data.to_csv(RUN_DIR / "constructed_monthly_returns.csv")


## 2. 时间与单位：当前面板的主窗口

month是连续月度PeriodIndex的收益区间标签，不是公告或成交时刻。当前构造数据已是小数：0.01表示1%，不能再除100。RF固定0.001也按同月连接，不是年利率或实际国库券观测。

下一单元检查2010-01至2019-12完整索引和120个月，展示当前构造主表开头及RF均值。原French发布/入库、信号与交易时钟不在当前输入中。

In [ ]:
main = data.loc["2010-01":"2019-12"].copy()
expected_main = pd.period_range("2010-01", "2019-12", freq="M", name="month")
assert main.index.equals(expected_main) and len(main) == 120

display(main.head())
print("主窗口：", main.index[0], "到", main.index[-1], "；", len(main), "个连续月度区间")
print("RF 同窗口月均（百分比 / 月）:", f"{main['RF'].mean() * 100:.4f}%")

## 3. 简单收益、对数收益与财富

下一单元是独立手算：初始100、+50%、−50%，不使用Seed101主表，也不使用French。无外部资金时W_t=W_{t-1}(1+R_t)，因此财富为100、150、75；回撤从运行峰值算。

对R_t>−1，g_t=ln(1+R_t)，时间上的Σg_t=ln(W_T/W_0)。对数跨时间可加，不可直接代替不同资产简单收益的资金加权。

In [ ]:
# 构造的两期例子只用于手算复利，不作为市场观测。
toy_returns = np.array([0.50, -0.50])
toy_wealth = wealth_index(toy_returns, initial_capital=100.0)
toy_drawdown = drawdown_path(toy_wealth)
print("构造收益:", toy_returns)
print("财富路径:", toy_wealth)
print("路径回撤:", toy_drawdown)
print("算术平均:", toy_returns.mean())
print("平均对数收益:", np.log1p(toy_returns).mean())
np.testing.assert_allclose(toy_wealth, [100.0, 150.0, 75.0])
np.testing.assert_allclose(toy_drawdown, [0.0, 0.0, -0.5])

接下来比较当前Seed101构造主窗口的月简单均值和平均对数收益。列名pct_per_month表示展示百分比，内部仍是小数。算术均值描述单期平均，对数均值描述这条路径的平均对数增长；财富仍用简单收益调用wealth_index，初始1不是价格。

In [ ]:
mean_returns = pd.DataFrame({
    "industry": [item["label_zh"] for item in config["industries"]],
    "mean_simple_return_pct_per_month": [main[item["column"]].mean() * 100 for item in config["industries"]],
    "mean_log_return_pct_per_month": [np.log1p(main[item["column"]]).mean() * 100 for item in config["industries"]],
})
display(mean_returns.round(4))

## 4. 当前构造面板的风险与增长指标

波动用月简单收益样本标准差(ddof=1)乘√12，是约定年度尺度，非年度复利风险恒等式。同月e_t=R_t−RF_t，Sharpe用√12乘超额均值/超额样本标准差，要求分母正；不是Alpha或显著性检验。

回撤为W_t/截至当期峰值−1，最大损失报正幅度；初始财富保留。CAGR=(W_end/W_start)^(12/n_months)−1使用整月跨度约定且无外部资金。下一单元的主表只来自当前构造输入。

In [ ]:
results = summarize_all_windows(data, config)
primary = results.loc[results["window_id"] == config["main_window"]].copy()
main_table = primary[[
    "asset_label_zh", "final_wealth_index", "cumulative_simple_return",
    "cagr_month_period_convention", "annualized_volatility_sqrt12_convention",
    "annualized_sharpe_sqrt12_convention", "maximum_drawdown_loss_fraction",
    "maximum_drawdown_peak_month", "maximum_drawdown_trough_month",
]].rename(columns={
    "asset_label_zh": "行业", "final_wealth_index": "期末财富指数",
    "cumulative_simple_return": "累计收益小数", "cagr_month_period_convention": "CAGR 小数",
    "annualized_volatility_sqrt12_convention": "年化波动率小数",
    "annualized_sharpe_sqrt12_convention": "Sharpe",
    "maximum_drawdown_loss_fraction": "最大回撤损失小数",
    "maximum_drawdown_peak_month": "峰值月", "maximum_drawdown_trough_month": "谷底月",
})
display(main_table.round(4))

### 如何阅读当前主窗口表

2010–2019为120个连续构造月。财富初值1，期末指数表示这条构造路径的相对财富，不是证券价格，也不是原报告的HiTec4.6270。CAGR压缩初末增长而隐藏路径；波动描述月度离散，回撤记录此前峰值到谷底，同月RF Sharpe加入机会成本。先比较各指标的问题，不能拿单个数选择未来赢家。

## 5. 固定分段：同一构造输入的窗口敏感性

下面三个60月区间全部取自Seed101构造面板。固定切片只揭示该输入的描述窗口敏感性，不是French市场留出期。每段重新以初始财富1归一化，回撤不延续前一段峰值。

In [ ]:
segments = results.loc[results["window_id"].isin(["2010_2014", "2015_2019", "2020_2024"]), [
    "window_id", "asset_label_zh", "cagr_month_period_convention",
    "annualized_volatility_sqrt12_convention", "annualized_sharpe_sqrt12_convention",
    "maximum_drawdown_loss_fraction",
]].copy()
segments["CAGR_pct"] = segments.pop("cagr_month_period_convention") * 100
segments["annualized_volatility_pct"] = segments.pop("annualized_volatility_sqrt12_convention") * 100
segments["maximum_drawdown_pct"] = segments.pop("maximum_drawdown_loss_fraction") * 100
display(segments.round(3))

上方是Seed101、180个月的构造方法演示表，不是原French市场报告。已接受Markdown记录的当前构造能源2015–2019 CAGR约+8.008%、Sharpe约+0.623；2020–2024约−0.262%和−0.073。这些记录未在本语言投影重新执行。保持正负与对应输入的解释，不能套用原报告能源负夏普等另一组历史结论。

## 6. 当前构造图的四个轴

下方新图写RUN_DIR/project01_dashboard_CONSTRUCTED.png，输入仍是Seed101面板。图覆盖2010–2024，灰色2020–2024为描述扩展。左上月末标签对初值1财富指数；右上月末对向后12月样本标准差乘√12，前11月为空；左下是相对历史峰值的负回撤；右下行业对月简单收益小数的箱线图。初始财富点不是价格，滚动风险不是未来预测。

In [ ]:
figure_path = make_project_figure(
    data, RUN_DIR / "project01_dashboard_CONSTRUCTED.png", config
)
print(figure_path)
display(Image(filename=str(figure_path)))

## 7. 对应证据和完整题答

当前Notebook演示构造收益的指标、窗口与图；[原历史报告](../../../../../projects/project01/research_report/report.md)另用French月度发布行业回报与同月RF，保留其版本、历史指标及许可边界。两者均不提供逐股OHLCV、历史PIT版本、精确发布/入库时刻、信号、成交、融资与成本；这些描述性收益和Sharpe不能建立可交易策略表现。

先做[六题](../../../../../projects/project01/exercises.md)再看[完整答案](../../../../../projects/project01/solutions.md)。第1题用 +10%/−10%，第2、5题读取原历史报告，不能用本构造表替换原答案。